In [1]:
from tqdm.notebook import tqdm
import polars as pl

# wags-llm imports
from wags_llm.cache import InMemoryCache
from wags_llm.client import BedrockClaudeJsonClient
from wags_llm.prompts import build_empty_registry
from wags_llm.services import StructuredTaskRunner

# dgiLIT Pydantic Models
from dgilit_models import InteractionExtractionResult
from dgilit_models import InteractionExtractionPrompt

# wags-llm
Wags-LLM exists to add structure to LLM prompt calls in a repeatable, explainable way. This is the first attempt at implementing this with a dgiLIT prompt. Other portions of the dgiLIT pipeline would happen before this step and as such, the supporting models and prompts in this branch will likely evolve as those are tuned up.  
  
Notable things to update:  
- pre-tagging of drugs and genes to submit alongside a prompt. 
- support for multiple interactions per one abstract or block of text
- support for chunking sections of a PMID (Abstract, Results, Conclusion)
- support for Pydantic model level sanity checks to reduce LLM usage where not needed (may involve pre-tagging or deterministic rule sets that would provide a `run_llm? TRUE FALSE` type evaluation)

In [2]:
MODEL_ID = "us.anthropic.claude-sonnet-4-6"
REGION_NAME = "us-east-1"
PROFILE_NAME = "dev-account"
MAX_TOKENS = 150

def build_llm_task_runner(
    model_id: str,
    region_name: str,
    profile_name: str,
    max_tokens: int,
    temperature: float,
) -> StructuredTaskRunner:
    """Build LLM interaction extraction task runner

    :param model_id: Bedrock model identifier.
    :param region_name: AWS region for the Bedrock runtime client.
    :param profile_name: AWS profile name.
    :param max_tokens: Maximum number of tokens to request from the model.
    :param temperature: Sampling temperature.
    :return: Configured structured task runner instance.
    """
    registry = build_empty_registry()
    registry.register(InteractionExtractionPrompt(version="v1"))
    llm_client = BedrockClaudeJsonClient(
        model_id=model_id,
        region_name=region_name,
        profile_name=profile_name,
        max_tokens=max_tokens,
        temperature=temperature,
    )
    cache = InMemoryCache()
    return StructuredTaskRunner(
        client=llm_client, prompt_registry=registry, cache=cache
    )

In [ ]:
def extraction_interactions(
    task_runner: StructuredTaskRunner,
    prompt: InteractionExtractionPrompt,
    context: str,
) -> InteractionExtractionResult:
    """Extract interactions between a drug and a gene given a context from a published biomedical article

    :param task_runner: Structured task runner used to execute the LLM prompt.
    :param prompt: Configured prompt instance for interaction extraction.
    :param context: The block of text to evaluate.
    :return: An Interaction Result with drug, gene, and interaction boolean.
    """


    payload = prompt.build_payload(
        context=context
    )

    try:
        task_result = task_runner.execute(
            prompt_name=prompt.name,
            prompt_version=prompt.version,
            payload=payload,
            response_model=InteractionExtractionResult,
        )

        return InteractionExtractionResult(
            drug=task_result.drug,
            gene=task_result.gene,
            interaction=task_result.interaction
        )

    except Exception as e:
        return InteractionExtractionResult(
            error_message=str(e)
        )

In [4]:
temperatures = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
df = pl.read_excel("2026-06-08-test1.xlsx") # file with pmids + context blocks

def run_experiments(df, temperatures, num_runs, prompt_version: str):
    """Run LLM annotation experiments across a range of temperatures and multiple runs per temperature, storing results for analysis.

    :param df: Input dataframe containing gene symbols and associated information.
    :param temperatures: List of temperature values to experiment with.
    :param num_runs: Number of runs to execute for each temperature setting.
    :param prompt_version: Version of the prompt template to use for annotation.
    :return: List of stored runs with LLM outputs and diagnostics for each temperature and run
    """
    stored_runs = []

    for temp in temperatures:
        for run_idx in range(num_runs):
            task_runner = build_llm_task_runner(
                MODEL_ID,
                REGION_NAME,
                PROFILE_NAME,
                MAX_TOKENS,
                temp,
            )

            prompt = InteractionExtractionPrompt(version=prompt_version)

            print(f"Running temp={temp}, run={run_idx + 1}")

            results = []

            for row in tqdm(
                df.iter_rows(named=True),
                total=df.height,
                desc=f"T={temp}, run={run_idx + 1}",
                leave=False,
            ):
                result = extraction_interactions(
                    task_runner=task_runner,
                    prompt=prompt,
                    context=row["context"],
                )

                if result.error_message:
                    raise RuntimeError(
                        f"""
                        Error in temp={temp}, run={run_idx + 1}
                        context={row["context"]}
                        error={result.error_message}
                        """
                    )
            # drug=task_result.drug,
            # gene=task_result.gene,
            # interaction=task_result.interaction

                results.append(
                    {
                        "drug": result.drug,
                        "gene": result.gene,
                        "interaction": result.interaction,
                    }
                )
            print("prompt_version being stored:", prompt_version)

            stored_runs.append(
                {
                    "run_idx": run_idx,
                    "prompt_version": prompt_version,
                    "temperature": temp,
                    "results": results,
                }
            )

            print(f"Done temp={temp}, run={run_idx + 1}")

    return stored_runs

NOTE: After running this once, temperature as a parameter really doesn't need to be included. Kinda just set it to 0 and forget about it tbh.

In [5]:
stored_runs = run_experiments(df, temperatures, num_runs=1, prompt_version='v1')

Running temp=0, run=1


T=0, run=1:   0%|          | 0/100 [00:00<?, ?it/s]

prompt_version being stored: v1
Done temp=0, run=1
Running temp=0.2, run=1


T=0.2, run=1:   0%|          | 0/100 [00:00<?, ?it/s]

prompt_version being stored: v1
Done temp=0.2, run=1
Running temp=0.4, run=1


...

prompt_version being stored: v1
Done temp=0.6, run=1
Running temp=0.8, run=1


T=0.8, run=1:   0%|          | 0/100 [00:00<?, ?it/s]

prompt_version being stored: v1
Done temp=0.8, run=1
Running temp=1.0, run=1


T=1.0, run=1:   0%|          | 0/100 [00:00<?, ?it/s]

prompt_version being stored: v1
Done temp=1.0, run=1


In [6]:
stored_runs

[{'run_idx': 0,
  'prompt_version': 'v1',
  'temperature': 0,
  'results': [{'drug': 'doxorubicin', 'gene': 'Bcl-2', 'interaction': False},
   {'drug': 'calcipotriol', 'gene': 'BCL2', 'interaction': True},
   {'drug': 'methylprednisolone sodium succinate',
    'gene': 'Bcl-2',
    'interaction': True},
   {'drug': 'compound 21', 'gene': 'Mcl-1', 'interaction': True},
   {'drug': 'ZD1839', 'gene': 'EGFR', 'interaction': True},
   {'drug': None, 'gene': None, 'interaction': None},
   {'drug': 'interferon-alpha', 'gene': 'FAS', 'interaction': True},
   {'drug': 'interferon', 'gene': 'BCL-2', 'interaction': True},
   {'drug': 'dolastatin 15', 'gene': 'bcl-2', 'interaction': True},
   {'drug': '5-fluorodeoxyuridine', 'gene': 'bcl-2', 'interaction': True},
   {'drug': 'hyperbaric oxygen', 'gene': 'Bcl-2', 'interaction': True},
   {'drug': 'hyperbaric oxygen', 'gene': 'Bcl-2', 'interaction': True},
   {'drug': 'Cyclophosphamide', 'gene': 'CPP-32', 'interaction': True},
   {'drug': 'PMSG', 'ge

### Save

Save the output from a wags-llm run. After running through this whole process but I think its good.

In [7]:
rows = []
for run in stored_runs:
    for result in run["results"]:
        rows.append({
            "run_idx": run["run_idx"],
            "prompt_version": run["prompt_version"],
            "temperature": run["temperature"],
            "drug": result["drug"],
            "gene": result["gene"],
            "interaction": result["interaction"],
        })

df_results = pl.DataFrame(rows)

df_results.write_csv("2026-06-08-interaction_predictions.csv")
df_results.write_parquet("2026-06-08-interaction_predictions.parquet")

df_results.head()

run_idx,prompt_version,temperature,drug,gene,interaction
i64,str,i64,str,str,bool
0,"""v1""",0,"""doxorubicin""","""Bcl-2""",false
0,"""v1""",0,"""calcipotriol""","""BCL2""",true
0,"""v1""",0,"""methylprednisolone sodium succ…","""Bcl-2""",true
0,"""v1""",0,"""compound 21""","""Mcl-1""",true
0,"""v1""",0,"""ZD1839""","""EGFR""",true
